# Task 5: Strategy Backtesting

## Objective
Validate the portfolio strategy by simulating its performance on historical data and comparing it against a benchmark. This helps understand whether the model-driven approach would have outperformed a simple passive strategy.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Data and Portfolio Configuration

In [ ]:
# Load processed data
tsla_data = pd.read_csv('../data/processed/tsla_processed.csv', index_col=0, parse_dates=True)
bnd_data = pd.read_csv('../data/processed/bnd_processed.csv', index_col=0, parse_dates=True)
spy_data = pd.read_csv('../data/processed/spy_processed.csv', index_col=0, parse_dates=True)

# Load optimal portfolio weights
with open('../data/processed/optimal_portfolio.json', 'r') as f:
    portfolio_config = json.load(f)

# Define backtesting period (last year: 2025-2026)
backtest_start = '2025-01-01'
backtest_end = '2026-01-15'

print("="*60)
print("BACKTESTING CONFIGURATION")
print("="*60)
print(f"Backtesting Period: {backtest_start} to {backtest_end}")
print(f"\nOptimal Portfolio Weights:")
print(f"  TSLA: {portfolio_config['TSLA_Weight']*100:.2f}%")
print(f"  BND: {portfolio_config['BND_Weight']*100:.2f}%")
print(f"  SPY: {portfolio_config['SPY_Weight']*100:.2f}%")

## 2. Prepare Backtesting Data

In [ ]:
# Filter data for backtesting period
tsla_backtest = tsla_data[(tsla_data.index >= backtest_start) & (tsla_data.index <= backtest_end)]
bnd_backtest = bnd_data[(bnd_data.index >= backtest_start) & (bnd_data.index <= backtest_end)]
spy_backtest = spy_data[(spy_data.index >= backtest_start) & (spy_data.index <= backtest_end)]

# Calculate daily returns
tsla_returns = tsla_backtest['Close'].pct_change().dropna()
bnd_returns = bnd_backtest['Close'].pct_change().dropna()
spy_returns = spy_backtest['Close'].pct_change().dropna()

# Align dates
common_dates = tsla_returns.index.intersection(bnd_returns.index).intersection(spy_returns.index)
tsla_returns = tsla_returns.loc[common_dates]
bnd_returns = bnd_returns.loc[common_dates]
spy_returns = spy_returns.loc[common_dates]

# Create returns dataframe
returns_df = pd.DataFrame({
    'TSLA': tsla_returns,
    'BND': bnd_returns,
    'SPY': spy_returns
})

print(f"Backtesting data shape: {returns_df.shape}")
print(f"Date range: {returns_df.index.min()} to {returns_df.index.max()}")
print(f"\nFirst few rows:")
print(returns_df.head())

## 3. Define Benchmark Portfolio

In [ ]:
# Benchmark: 60% SPY / 40% BND (balanced portfolio)
benchmark_weights = {
    'SPY': 0.60,
    'BND': 0.40,
    'TSLA': 0.00
}

print("="*60)
print("BENCHMARK PORTFOLIO")
print("="*60)
print(f"SPY: {benchmark_weights['SPY']*100:.0f}%")
print(f"BND: {benchmark_weights['BND']*100:.0f}%")
print(f"TSLA: {benchmark_weights['TSLA']*100:.0f}%")

## 4. Simulate Strategy Performance

In [ ]:
# Strategy portfolio weights
strategy_weights = {
    'TSLA': portfolio_config['TSLA_Weight'],
    'BND': portfolio_config['BND_Weight'],
    'SPY': portfolio_config['SPY_Weight']
}

# Calculate portfolio returns (simple hold strategy - no rebalancing)
strategy_returns = (
    returns_df['TSLA'] * strategy_weights['TSLA'] +
    returns_df['BND'] * strategy_weights['BND'] +
    returns_df['SPY'] * strategy_weights['SPY']
)

benchmark_returns = (
    returns_df['SPY'] * benchmark_weights['SPY'] +
    returns_df['BND'] * benchmark_weights['BND']
)

# Calculate cumulative returns
strategy_cumulative = (1 + strategy_returns).cumprod()
benchmark_cumulative = (1 + benchmark_returns).cumprod()

# Create performance dataframe
performance_df = pd.DataFrame({
    'Strategy': strategy_cumulative,
    'Benchmark': benchmark_cumulative,
    'Strategy_Daily_Return': strategy_returns,
    'Benchmark_Daily_Return': benchmark_returns
}, index=returns_df.index)

print("="*60)
print("PORTFOLIO SIMULATION COMPLETE")
print("="*60)
print(f"\nStrategy Final Value: {strategy_cumulative.iloc[-1]:.4f}")
print(f"Benchmark Final Value: {benchmark_cumulative.iloc[-1]:.4f}")
print(f"\nStrategy Total Return: {(strategy_cumulative.iloc[-1] - 1) * 100:.2f}%")
print(f"Benchmark Total Return: {(benchmark_cumulative.iloc[-1] - 1) * 100:.2f}%")

## 5. Calculate Performance Metrics

In [ ]:
# Calculate performance metrics
def calculate_performance_metrics(returns, cumulative_returns):
    """Calculate comprehensive performance metrics"""
    # Total return
    total_return = (cumulative_returns.iloc[-1] - 1) * 100
    
    # Annualized return
    num_days = len(returns)
    num_years = num_days / 252
    annualized_return = ((cumulative_returns.iloc[-1] ** (1/num_years)) - 1) * 100
    
    # Volatility (annualized)
    annualized_vol = returns.std() * np.sqrt(252) * 100
    
    # Sharpe Ratio (assuming risk-free rate of 2%)
    risk_free_rate = 0.02
    excess_returns = returns - (risk_free_rate / 252)
    sharpe_ratio = np.sqrt(252) * excess_returns.mean() / returns.std()
    
    # Maximum drawdown
    running_max = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    
    return {
        'Total_Return_%': total_return,
        'Annualized_Return_%': annualized_return,
        'Annualized_Volatility_%': annualized_vol,
        'Sharpe_Ratio': sharpe_ratio,
        'Max_Drawdown_%': max_drawdown
    }

# Calculate metrics for both portfolios
strategy_metrics = calculate_performance_metrics(
    performance_df['Strategy_Daily_Return'],
    performance_df['Strategy']
)

benchmark_metrics = calculate_performance_metrics(
    performance_df['Benchmark_Daily_Return'],
    performance_df['Benchmark']
)

# Create comparison table
metrics_comparison = pd.DataFrame({
    'Strategy': strategy_metrics,
    'Benchmark': benchmark_metrics
})

print("="*60)
print("PERFORMANCE METRICS COMPARISON")
print("="*60)
print(metrics_comparison.round(4))

# Calculate outperformance
outperformance = strategy_metrics['Total_Return_%'] - benchmark_metrics['Total_Return_%']
print(f"\nStrategy Outperformance: {outperformance:.2f} percentage points")

## 6. Visualize Performance

In [ ]:
# Plot cumulative returns comparison
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Cumulative returns
axes[0].plot(performance_df.index, performance_df['Strategy'], 
             label='Strategy Portfolio', color='blue', linewidth=2)
axes[0].plot(performance_df.index, performance_df['Benchmark'], 
             label='Benchmark (60% SPY / 40% BND)', color='orange', linewidth=2)
axes[0].set_title('Cumulative Returns Comparison', fontsize=16, fontweight='bold')
axes[0].set_ylabel('Cumulative Return', fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: Drawdown
strategy_running_max = performance_df['Strategy'].expanding().max()
strategy_drawdown = (performance_df['Strategy'] - strategy_running_max) / strategy_running_max * 100

benchmark_running_max = performance_df['Benchmark'].expanding().max()
benchmark_drawdown = (performance_df['Benchmark'] - benchmark_running_max) / benchmark_running_max * 100

axes[1].fill_between(performance_df.index, strategy_drawdown, 0, 
                     alpha=0.3, color='blue', label='Strategy Drawdown')
axes[1].fill_between(performance_df.index, benchmark_drawdown, 0, 
                     alpha=0.3, color='orange', label='Benchmark Drawdown')
axes[1].set_title('Drawdown Analysis', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Drawdown (%)', fontsize=12)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/backtest_performance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create performance metrics visualization
fig, ax = plt.subplots(figsize=(12, 8))

metrics_to_plot = ['Total_Return_%', 'Annualized_Return_%', 'Sharpe_Ratio']
x_pos = np.arange(len(metrics_to_plot))
width = 0.35

strategy_values = [strategy_metrics[m] for m in metrics_to_plot]
benchmark_values = [benchmark_metrics[m] for m in metrics_to_plot]

bars1 = ax.bar(x_pos - width/2, strategy_values, width, 
               label='Strategy', color='blue', alpha=0.7)
bars2 = ax.bar(x_pos + width/2, benchmark_values, width, 
               label='Benchmark', color='orange', alpha=0.7)

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Performance Metrics Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Total Return (%)', 'Annualized Return (%)', 'Sharpe Ratio'])
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../data/processed/performance_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Save Results and Conclusion

In [ ]:
# Save backtesting results
performance_df.to_csv('../data/processed/backtest_results.csv')

# Save metrics
backtest_summary = {
    'Backtest_Period_Start': backtest_start,
    'Backtest_Period_End': backtest_end,
    'Strategy_Metrics': strategy_metrics,
    'Benchmark_Metrics': benchmark_metrics,
    'Outperformance_Pct': float(outperformance)
}

with open('../data/processed/backtest_summary.json', 'w') as f:
    json.dump(backtest_summary, f, indent=4)

print("Backtesting results saved successfully!")
print(f"\nResults saved to:")
print(f"  - ../data/processed/backtest_results.csv")
print(f"  - ../data/processed/backtest_summary.json")

## 8. Conclusion and Reflection

### Strategy Performance Assessment

**Did the strategy outperform the benchmark?**

[Based on the results, provide a clear answer: YES/NO and by how much]

### Key Findings

1. **Total Return**: The strategy achieved [X]% total return compared to the benchmark's [Y]%
2. **Risk-Adjusted Performance**: The strategy's Sharpe Ratio of [X] compares to the benchmark's [Y]
3. **Volatility**: The strategy exhibited [X]% annualized volatility vs. [Y]% for the benchmark
4. **Maximum Drawdown**: The strategy's maximum drawdown was [X]% vs. [Y]% for the benchmark

### Implications

**What does this backtest suggest about the viability of the model-driven approach?**

The backtest results indicate that [provide analysis based on actual results]:

- If the strategy outperformed: The model-driven approach shows promise, suggesting that incorporating forecasted returns for TSLA can enhance portfolio performance.

- If the strategy underperformed: This highlights the challenges of forecasting in financial markets and the importance of considering model uncertainty.

### Limitations of This Backtest

1. **Look-Ahead Bias**: The portfolio weights were determined using information that would not have been available at the start of the backtesting period (the forecast was generated using data up to 2026).

2. **No Transaction Costs**: The simulation assumes no trading costs, which would reduce actual returns.

3. **No Rebalancing**: The strategy holds fixed weights throughout the period. In practice, periodic rebalancing would be necessary.

4. **Limited Time Period**: One year of backtesting may not be sufficient to draw definitive conclusions about long-term performance.

5. **Model Assumptions**: The forecast model assumes that past patterns will continue, which may not hold in changing market conditions.

6. **Benchmark Selection**: The 60/40 benchmark may not be the most appropriate comparison for all investors.

### Recommendations

Based on this backtest:

1. [Recommendation 1]
2. [Recommendation 2]
3. [Recommendation 3]

**Overall Assessment**: [Provide a final assessment of whether the model-driven approach is viable and under what conditions it might be most effective]